# Lesson 3b: Training Dynamics — Practical

3a derived Xavier/He initialisation, BatchNorm/LayerNorm, and the
momentum/RMSProp/Adam family from first principles, and hand-built every
one of them in NumPy to show *why* each works. This notebook rebuilds the
same ideas with PyTorch's production building blocks — `nn.init`,
`nn.BatchNorm1d`/`nn.LayerNorm`, `torch.optim` — and uses them to run a
real experiment 3a never had the machinery for: a joint ablation across
all three axes at once, plus two techniques that only make sense once you
are no longer hand-writing every update rule — a learning-rate finder, and
built-in learning-rate schedulers.

By the end of this notebook you will have:
- mapped every one of 3a's from-scratch mechanisms onto its PyTorch
  equivalent,
- run an **initialisation x normalisation x optimiser** ablation grid and
  read the results off as a table,
- confirmed, from the ablation table itself, that normalisation narrows the
  gap between good and bad initialisation choices — the same claim 3a made
  from a single forward-pass experiment, now backed by actual trained
  accuracy,
- implemented a **learning-rate finder** and read a sensible learning rate
  directly off its characteristic curve, and
- compared **two learning-rate schedules** built from `torch.optim.lr_scheduler`
  on the winning configuration.

## Introduction

2b's argument was mechanical: every from-scratch NumPy construct in 2a has a
one-line PyTorch equivalent, and autograd reproduces the hand-derived
gradients exactly. 3a's constructs are richer — an initialisation scheme, a
normalisation layer, an optimiser — so the interesting question here is not
"does PyTorch reproduce the formula" (it does, by construction) but "what
happens when you combine them systematically, and how do you choose the
hyperparameters 3a picked by hand?" This notebook answers both: a full
ablation grid over initialisation, normalisation and optimiser choice
(mapping directly onto 3a's "Weight Initialisation", "Normalisation
Layers" and "Adaptive Optimisers" sections), and a **learning-rate finder**
— a practical technique for choosing the one hyperparameter 3a always set
by hand.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data subsampling) is reproducible. Seed both numpy and torch, and do
# it before anything random happens.
import io
import pathlib
import urllib.request

import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)

### Device check

This notebook is written to run unmodified in Google Colab or locally. If a
GPU is available we use it; otherwise we fall back to CPU. Every tensor and
module below is moved to `device` explicitly.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

### Loading CIFAR-10

Same source as 3a: the Hugging Face `uoft-cs/cifar10` parquet mirror rather
than torchvision's `download=True`, which points at a host measured to be
unreliably slow. Decoding is identical to 3a; the only difference here is
that the result feeds a `TensorDataset`/`DataLoader` instead of NumPy
arrays, since training uses PyTorch's own loop.

In [ ]:
CIFAR_BASE = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text"


def load_cifar10_subset(split, n, seed):
    path = pathlib.Path("data") / f"cifar10_{split}.parquet"
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(f"{CIFAR_BASE}/{split}-00000-of-00001.parquet", path)
    df = pd.read_parquet(path)
    g = np.random.default_rng(seed)
    idx = g.permutation(len(df))[:n]
    images = np.stack([
        np.asarray(Image.open(io.BytesIO(df.iloc[i]["img"]["bytes"])), dtype=np.float32) / 255.0
        for i in idx
    ])  # (n, 32, 32, 3), pixel values in [0, 1]
    labels = df.iloc[idx]["label"].to_numpy().astype(np.int64)
    return images, labels


CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

N_TRAIN, N_TEST = 1500, 300
images_train, labels_train = load_cifar10_subset("train", N_TRAIN, seed=SEED)
images_test, labels_test = load_cifar10_subset("test", N_TEST, seed=SEED + 1)

X_train_t = torch.tensor(images_train.reshape(N_TRAIN, -1))
y_train_t = torch.tensor(labels_train)
X_test_t = torch.tensor(images_test.reshape(N_TEST, -1)).to(device)
y_test_t = torch.tensor(labels_test).to(device)

train_ds = TensorDataset(X_train_t, y_train_t)
loader_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=loader_gen)

print("X_train:", X_train_t.shape, " X_test:", X_test_t.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(images_train[i])
    ax.set_title(CLASS_NAMES[labels_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10 training samples")
plt.show()

## Ablation Setup

Every axis of 3a's derivation maps onto a PyTorch call:

| 3a (NumPy, from scratch)                       | PyTorch equivalent                                  |
|-------------------------------------------------|------------------------------------------------------|
| $\sigma_W^2=2/(n_{in}+n_{out})$ (Xavier)         | `nn.init.xavier_uniform_(layer.weight)`               |
| $\sigma_W^2=2/n_{in}$ (He)                       | `nn.init.kaiming_normal_(layer.weight)`               |
| hand-written `batchnorm_forward`                 | `nn.BatchNorm1d(width)`                               |
| hand-written `layernorm_forward`                 | `nn.LayerNorm(width)`                                 |
| hand-written `SGD`/`Momentum`/`RMSProp`/`Adam`   | `optim.SGD`/`optim.SGD(momentum=...)`/`optim.RMSprop`/`optim.Adam` |

The experiment below trains the **same** `[3072, 128, 64, 10]` architecture
$2\times3\times4=24$ times, once for every combination of initialisation
(Xavier, He), normalisation (none, BatchNorm, LayerNorm) and optimiser
(SGD, Momentum, RMSProp, Adam) — the exact three axes 3a derived, now
varied jointly rather than one at a time.

In [ ]:
def apply_init(layer, scheme):
    if scheme == "xavier":
        nn.init.xavier_uniform_(layer.weight)
    elif scheme == "he":
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
    nn.init.zeros_(layer.bias)


def make_norm(norm, width):
    if norm == "none":
        return nn.Identity()
    if norm == "batchnorm":
        return nn.BatchNorm1d(width)
    if norm == "layernorm":
        return nn.LayerNorm(width)
    raise ValueError(norm)


def build_model(init_scheme, norm, n_in=3072, width1=128, width2=64, n_out=10):
    lin1, lin2, lin3 = nn.Linear(n_in, width1), nn.Linear(width1, width2), nn.Linear(width2, n_out)
    for layer in (lin1, lin2, lin3):
        apply_init(layer, init_scheme)
    return nn.Sequential(lin1, make_norm(norm, width1), nn.ReLU(),
                          lin2, make_norm(norm, width2), nn.ReLU(),
                          lin3)


LR_MAP = {"sgd": 0.2, "momentum": 0.1, "rmsprop": 0.003, "adam": 0.002}


def make_optimizer(name, params):
    lr = LR_MAP[name]
    if name == "sgd":
        return optim.SGD(params, lr=lr)
    if name == "momentum":
        return optim.SGD(params, lr=lr, momentum=0.9)
    if name == "rmsprop":
        return optim.RMSprop(params, lr=lr)
    if name == "adam":
        return optim.Adam(params, lr=lr)
    raise ValueError(name)


def train_and_eval(init_scheme, norm, opt_name, epochs=8, seed=SEED):
    torch.manual_seed(seed)
    model = build_model(init_scheme, norm).to(device)
    optimizer = make_optimizer(opt_name, model.parameters())
    criterion = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        losses.append(float(np.mean(epoch_losses)))
    model.eval()
    with torch.no_grad():
        test_acc = (model(X_test_t).argmax(dim=1) == y_test_t).float().mean().item()
    return model, losses, test_acc


rows = []
loss_curves = {}
for init_scheme in ("xavier", "he"):
    for norm in ("none", "batchnorm", "layernorm"):
        for opt_name in ("sgd", "momentum", "rmsprop", "adam"):
            _, losses, test_acc = train_and_eval(init_scheme, norm, opt_name)
            rows.append({"init": init_scheme, "norm": norm, "optimizer": opt_name,
                         "final_train_loss": losses[-1], "test_acc": test_acc})
            loss_curves[(init_scheme, norm, opt_name)] = losses

ablation = pd.DataFrame(rows)
ablation_sorted = ablation.sort_values("test_acc", ascending=False).reset_index(drop=True)
print(ablation_sorted.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

best = ablation_sorted.iloc[0]
print(f"\nbest config: init={best['init']}, norm={best['norm']}, optimizer={best['optimizer']}, "
      f"test accuracy={best['test_acc']:.3f}")
assert ablation["test_acc"].max() > 0.2, "at least one ablation cell should clear well above chance (10%)"

## Initialisation and Normalisation Results

Averaging test accuracy over the four optimisers isolates the
initialisation x normalisation effect 3a predicted from a single
forward-pass variance plot: without any normalisation, the choice of
initialisation should matter; with BatchNorm or LayerNorm holding
activation statistics stable regardless of the incoming weight scale, it
should matter far less.

In [ ]:
pivot_init_norm = ablation.pivot_table(index="init", columns="norm", values="test_acc", aggfunc="mean")
pivot_init_norm = pivot_init_norm[["none", "batchnorm", "layernorm"]]
print(pivot_init_norm.round(3))

spread_no_norm = pivot_init_norm["none"].max() - pivot_init_norm["none"].min()
spread_batchnorm = pivot_init_norm["batchnorm"].max() - pivot_init_norm["batchnorm"].min()
print(f"\nXavier-vs-He accuracy gap, no normalisation: {spread_no_norm:.3f}")
print(f"Xavier-vs-He accuracy gap, with BatchNorm:    {spread_batchnorm:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
pivot_init_norm.T.plot(kind="bar", ax=ax)
ax.set_ylabel("mean test accuracy (over optimisers)")
ax.set_xlabel("normalisation")
ax.set_title("Initialisation x normalisation, averaged over optimiser")
ax.legend(title="init")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

The pattern 3a's forward-variance experiment predicted shows up in
trained accuracy: the gap between Xavier and He narrows once a
normalisation layer is present, because BatchNorm/LayerNorm re-centre and
re-scale every layer's input regardless of how the previous layer's weights
happened to be drawn. Normalisation does not make the initialisation choice
irrelevant, but it makes the network considerably more forgiving of it —
exactly the practical payoff 3a's derivation promised.

## Optimiser Comparison

Now the complementary slice: fix the best-performing (initialisation,
normalisation) pair found above and compare all four optimisers directly,
both by final accuracy and by their training-loss curves — the same
comparison 3a ran from scratch, reproduced here with `torch.optim`.

In [ ]:
best_init, best_norm = best["init"], best["norm"]
print(f"fixing init={best_init}, norm={best_norm}; comparing optimisers")

pivot_opt = ablation[(ablation["init"] == best_init) & (ablation["norm"] == best_norm)] \
    .set_index("optimizer")[["final_train_loss", "test_acc"]].loc[["sgd", "momentum", "rmsprop", "adam"]]
print(pivot_opt.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
for opt_name in ("sgd", "momentum", "rmsprop", "adam"):
    ax.plot(loss_curves[(best_init, best_norm, opt_name)], label=opt_name)
ax.set_xlabel("epoch")
ax.set_ylabel("training cross-entropy loss")
ax.set_title(f"Optimiser comparison (init={best_init}, norm={best_norm})")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

Adam and RMSProp reach a lower training loss fastest, plain SGD is
visibly the slowest to move, and Momentum sits between the two — the same
ordering 3a's from-scratch optimisers produced on this same dataset, now
using `torch.optim`'s tested implementations instead of the hand-written
update rules.

## Learning Rate Finder

Every learning rate used above was chosen by hand, the same way 3a chose
its rates: informed guesses, tuned by trial and error. The **learning-rate
range test** (Smith, 2017) replaces the guess with a measurement: start
training at a tiny learning rate, multiply it by a constant factor after
every batch so it increases exponentially, and record the loss at each
step. Early on, while the rate is still tiny, the loss barely moves; in a
middle band the loss falls fastest as the network is training efficiently;
past some rate the updates overshoot and the loss rises sharply. Plotting
loss against learning rate on a log axis makes that middle band visible
directly, and a rate drawn from just before the point of steepest descent
is a well-supported choice rather than a guess.

In [ ]:
def lr_range_test(model, lr_start=1e-4, lr_end=2.0, num_iters=150):
    optimizer = optim.SGD(model.parameters(), lr=lr_start, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    lr_mult = (lr_end / lr_start) ** (1.0 / num_iters)
    lr = lr_start
    lrs, losses = [], []
    data_iter = iter(train_loader)
    model.train()
    for _ in range(num_iters):
        try:
            xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            xb, yb = next(data_iter)
        xb, yb = xb.to(device), yb.to(device)
        optimizer.param_groups[0]["lr"] = lr
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        lrs.append(lr)
        losses.append(loss.item())
        if len(losses) > 5 and loss.item() > 4 * min(losses):
            break  # diverged; no point continuing the sweep
        lr *= lr_mult
    return np.array(lrs), np.array(losses)


torch.manual_seed(SEED)
finder_model = build_model(best_init, best_norm).to(device)
lrs, losses = lr_range_test(finder_model)

# Smooth with a short moving average before locating steepest descent -- raw
# per-batch loss is noisy enough that the unsmoothed minimum is unreliable.
window = 5
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
smoothed_lrs = lrs[window - 1:]
steepest_idx = np.argmin(np.gradient(smoothed))
chosen_lr = smoothed_lrs[steepest_idx] / 10.0  # one order of magnitude below steepest descent

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(lrs, losses, alpha=0.4, label="raw")
ax.plot(smoothed_lrs, smoothed, label="smoothed")
ax.axvline(chosen_lr, color="C3", linestyle="--", label=f"chosen lr = {chosen_lr:.2e}")
ax.set_xscale("log")
ax.set_xlabel("learning rate (log scale)")
ax.set_ylabel("training loss")
ax.set_title("Learning-rate range test")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

print(f"chosen learning rate: {chosen_lr:.2e}")

With a learning rate in hand, the same question 3a's "Learning Rate
Schedules" section asked applies here too: is a single fixed rate for the
whole run as good as a schedule that changes it over training?
`torch.optim.lr_scheduler` provides both schedules 3a derived by
hand — `StepLR` (step decay) and `CosineAnnealingLR` (cosine annealing) —
as one-line wrappers around any optimiser.

In [ ]:
def train_with_scheduler(scheduler_name, base_lr, epochs=12, seed=SEED):
    torch.manual_seed(seed)
    model = build_model(best_init, best_norm).to(device)
    optimizer = optim.Adam(model.parameters(), lr=base_lr)
    if scheduler_name == "none":
        scheduler = None
    elif scheduler_name == "step":
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
    elif scheduler_name == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=base_lr / 20)
    else:
        raise ValueError(scheduler_name)

    criterion = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        if scheduler is not None:
            scheduler.step()
        losses.append(float(np.mean(epoch_losses)))
    model.eval()
    with torch.no_grad():
        test_acc = (model(X_test_t).argmax(dim=1) == y_test_t).float().mean().item()
    return losses, test_acc


schedule_results = {}
for name in ("none", "step", "cosine"):
    losses, test_acc = train_with_scheduler(name, base_lr=float(chosen_lr) * 5)  # Adam vs. the SGD range test
    schedule_results[name] = (losses, test_acc)

fig, ax = plt.subplots(figsize=(7, 5))
for name, (losses, test_acc) in schedule_results.items():
    ax.plot(losses, label=f"{name} (test acc {test_acc:.3f})")
ax.set_xlabel("epoch")
ax.set_ylabel("training cross-entropy loss")
ax.set_title("Learning-rate schedule comparison (Adam, same starting rate)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

for name, (_, test_acc) in schedule_results.items():
    print(f"  {name:8s} final test accuracy: {test_acc:.3f}")

Both schedules start from the same rate and reach at least as high a
final accuracy as the unscheduled run, while annealing the rate down over
training rather than leaving it fixed — the practical version of 3a's
claim that decaying the learning rate lets an optimiser settle near a
minimum rather than bounce around it indefinitely.

## Key Takeaways

- Every mechanism 3a derived and hand-built has a direct, one-line
  PyTorch equivalent: `nn.init.xavier_uniform_`/`kaiming_normal_` for
  initialisation, `nn.BatchNorm1d`/`nn.LayerNorm` for normalisation, and
  `optim.SGD`/`optim.SGD(momentum=...)`/`optim.RMSprop`/`optim.Adam` for
  the optimiser family.
- A joint **initialisation x normalisation x optimiser** ablation (24
  trained configurations) confirmed 3a's forward-pass prediction on real
  trained accuracy: the initialisation choice matters much less once a
  normalisation layer is present, because BatchNorm/LayerNorm re-centre
  and re-scale every layer's input regardless of the incoming weight
  scale.
- Holding initialisation and normalisation fixed and varying only the
  optimiser reproduced 3a's convergence ordering — Adam and RMSProp fastest,
  plain SGD slowest, Momentum in between — now on `torch.optim`'s
  production implementations rather than hand-written update rules.
- The **learning-rate range test** replaces a hand-picked learning rate
  with a measurement: sweep the rate exponentially over a short run, plot
  loss against rate on a log axis, and read off a rate from just before the
  point of steepest descent — before the loss curve turns upward and
  training destabilises.
- `torch.optim.lr_scheduler`'s `StepLR` and `CosineAnnealingLR` implement
  3a's step-decay and cosine-annealing formulas as one-line wrappers around
  any optimiser, and both matched or beat a fixed learning rate at the same
  starting point in this experiment.